# **Handling Mixed Variables**

## **Topic Roadmap**

- **1. Introduction & Setup**
  - 1.1 Imports & Synthetic Data Generation
- **2. Case 1: Simple Mixed Column (Numbers and String Labels)**
  - 2.1 Numeric Extraction via `pd.to_numeric
  - 2.2 Categorical Extraction via `np.where
- **3. Case 2: Alphanumeric String Column (Cabin Code)**
  - 3.1 Regex Numerical Extraction
  - 3.2 String Index Categorical Extraction
- **4. Case 3: Space-Separated Ticket Codes**
  - 4.1 Splitting String Tokens
  - 4.2 Categorical & Numerical Separation
- **5. Key Revision Notes**

## **1. Introduction & Setup**

Mixed variables contain both numbers and categories within the same column or string. Machine learning models cannot interpret these mixed types directly. 

This section sets up the required libraries and creates a standalone dataset representing common mixed-variable scenarios.

In [1]:
import numpy as np
import pandas as pd

# Create synthetic DataFrame to ensure notebook reproducibility
data = {
    'number': ['1', '2', 'A', '3', 'B', '4', 'C'],
    'Cabin': ['C85', 'C123', 'E46', 'G6', 'C148', np.nan, 'A16'],
    'Ticket': ['A/5 21171', 'PC 17599', 'STON/O2. 3101282', '113803', '373450', 'PP 9549', 'A/4 4887']
}

df = pd.DataFrame(data)
df.head()

,number,Cabin,Ticket
0,1,C85,A/5 21171
1,2,C123,PC 17599
2,A,E46,STON/O2. 3101282
3,3,G6,113803
4,B,C148,373450


## **2. Case 1: Simple Mixed Column**

### **Separating Numeric and String Values**

When a single column contains distinct numbers and category strings, extract the numeric component using `pd.to_numeric` with `errors='coerce'`. Non-numeric items become `NaN`. The remaining string labels are stored in a parallel categorical column.

In [2]:
case1_df = df.copy()

# Extract numeric components
case1_df['number_numerical'] = pd.to_numeric(
    case1_df['number'], 
    errors='coerce', 
    downcast='integer'
)

# Extract categorical components
case1_df['number_categorical'] = np.where(
    case1_df['number_numerical'].isnull(), 
    case1_df['number'], 
    np.nan
)

case1_df[['number', 'number_numerical', 'number_categorical']]

,number,number_numerical,number_categorical
0,1,1.0,NaN
1,2,2.0,NaN
2,A,NaN,A
3,3,3.0,NaN
4,B,NaN,B
5,4,4.0,NaN
6,C,NaN,C


## **3. Case 2: Alphanumeric String Column**

### **Regex Extraction for Structural Codes (e.g., Cabin Numbers)**

Alphanumeric strings (e.g., `C85`) combine deck letters and room numbers into one token. Use regular expressions (`str.extract('(\d+)')`) to parse digits into a numeric variable and string indexing (`str[0]`) to isolate prefix categories.

In [3]:
cabin_df = df.copy()

# Extract numeric part using regex
cabin_df['cabin_num'] = cabin_df['Cabin'].str.extract('(\d+)')

# Extract categorical prefix (first letter)
cabin_df['cabin_cat'] = cabin_df['Cabin'].str[0]

cabin_df[['Cabin', 'cabin_num', 'cabin_cat']]

,Cabin,cabin_num,cabin_cat
0,C85,85,C
1,C123,123,C
2,E46,46,E
3,G6,6,G
4,C148,148,C
5,NaN,NaN,NaN
6,A16,16,A


## **4. Case 3: Space-Separated Ticket Codes**

### **Token Splitting for Compound Strings**

Compound ticket codes may consist of prefix codes and numeric sequences (e.g., `A/5 21171`) or purely numeric sequences (`113803`). Parse space-delimited elements to isolate prefix strings from numerical values.

In [4]:
ticket_df = df.copy()

# Extract last element as number
ticket_df['ticket_num'] = ticket_df['Ticket'].apply(lambda s: s.split()[-1])
ticket_df['ticket_num'] = pd.to_numeric(
    ticket_df['ticket_num'], 
    errors='coerce', 
    downcast='integer'
)

# Extract first element as prefix (if non-numeric)
ticket_df['ticket_cat'] = ticket_df['Ticket'].apply(lambda s: s.split()[0])
ticket_df['ticket_cat'] = np.where(
    ticket_df['ticket_cat'].str.isdigit(), 
    np.nan, 
    ticket_df['ticket_cat']
)

ticket_df[['Ticket', 'ticket_num', 'ticket_cat']]

,Ticket,ticket_num,ticket_cat
0,A/5 21171,21171,A/5
1,PC 17599,17599,PC
2,STON/O2. 3101282,3101282,STON/O2.
3,113803,113803,NaN
4,373450,373450,NaN
5,PP 9549,9549,PP
6,A/4 4887,4887,A/4


## **Key Revision Notes**

- **Definition**: Mixed variables contain a blend of numerical numbers and categorical labels within the same column or string pattern.
- **Parsing Simple Mixed Values**: Use `pd.to_numeric(series, errors='coerce')` to parse values. Non-numerics convert to `NaN`, allowing easy isolation via `np.where()`.
- **Regex Parsing**: Use `.str.extract('(\d+)')` to capture continuous digit groups embedded in alphanumeric strings.
- **String Parsing**: Use `.str.split()` or indexing (`.str[0]`) to split prefix codes or prefix letters from structural identifiers.
- **Post-Processing**: After separating mixed features into numerical and categorical columns, apply standard missing value imputation and categorical encoding techniques to each derived feature.